# Sentimento de resenhas de filme

O alvo é `positive` / `negative`. O modelo é Naive Bayes sobre as 1000 palavras
mais frequentes, depois da limpeza. Não há ganho em GPU: o custo está no texto,
não na álgebra do classificador.

A limpeza original percorria cada caractere em Python e recriava a lista de
stopwords a cada resenha. Aqui uma expressão regular tira HTML e símbolos,
e o stemmer e as stopwords são criados uma vez. O resultado do stemmer não é
idêntico token a token ao `word_tokenize` do NLTK, então a acurácia pode
deslocar um pouco em relação ao notebook antigo.

In [1]:
import pickle
import re
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB, GaussianNB, MultinomialNB

nltk.download("stopwords", quiet=True)
STOP = set(stopwords.words("english"))
STEMMER = SnowballStemmer("english")
TAG = re.compile(r"<.*?>")
SIMBOLO = re.compile(r"[^0-9A-Za-z]+")


def limpar(texto):
    texto = TAG.sub(" ", texto)
    texto = SIMBOLO.sub(" ", texto).lower()
    return " ".join(STEMMER.stem(w) for w in texto.split() if w not in STOP)


df = pd.read_csv("dataset.csv")
df["sentiment"] = df["sentiment"].map({"positive": 1, "negative": 0})
df["review"] = df["review"].map(limpar)
print(df["sentiment"].value_counts())
print(df["review"].iloc[0][:240])

sentiment
1    25000
0    25000
Name: count, dtype: int64
one review mention watch 1 oz episod hook right exact happen first thing struck oz brutal unflinch scene violenc set right word go trust show faint heart timid show pull punch regard drug sex violenc hardcor classic use word call oz nicknam


## Três Naive Bayes

O vetorizador aprende o vocabulário só no treino. GaussianNB trata a contagem
como número contínuo. MultinomialNB é o modelo de contagem de palavras.
BernoulliNB olha só se a palavra apareceu ou não.

O arquivo gravado é o de maior AUC no teste, não um nome fixo.

In [2]:
x_treino, x_teste, y_treino, y_teste = train_test_split(
    df["review"], df["sentiment"], test_size=0.2, random_state=0
)
vetorizador = CountVectorizer(max_features=1000)
X_treino = vetorizador.fit_transform(x_treino)
X_teste = vetorizador.transform(x_teste)
print(X_treino.shape, X_teste.shape)

modelos = {
    "GaussianNB": (GaussianNB(), X_treino.toarray(), X_teste.toarray()),
    "MultinomialNB": (MultinomialNB(), X_treino, X_teste),
    "BernoulliNB": (BernoulliNB(), X_treino, X_teste),
}
ranking = []
for nome, (modelo, treino_x, teste_x) in modelos.items():
    modelo.fit(treino_x, y_treino)
    auc = roc_auc_score(y_teste, modelo.predict_proba(teste_x)[:, 1])
    acc = accuracy_score(y_teste, modelo.predict(teste_x))
    print("%s  acurácia %.3f  AUC %.3f" % (nome, acc, auc))
    ranking.append((auc, nome, modelo))

auc, nome, melhor = max(ranking, key=lambda item: item[0])
print("escolhido:", nome, "AUC", round(auc, 3))
pickle.dump({"modelo": melhor, "vetorizador": vetorizador, "nome": nome}, open("modelo_sentimento.pkl", "wb"))

(40000, 1000) (10000, 1000)
GaussianNB  acurácia 0.790  AUC 0.862
MultinomialNB  acurácia 0.826  AUC 0.899
BernoulliNB  acurácia 0.832  AUC 0.908
escolhido: BernoulliNB AUC 0.908


## Texto novo

A mesma limpeza e o mesmo vetorizador. 1 é sentimento positivo, 0 é negativo.

In [3]:
pacote = pickle.load(open("modelo_sentimento.pkl", "rb"))
texto = """This is probably the fastest-paced and most action-packed of the German Edgar Wallace krimi series."""
vetor = pacote["vetorizador"].transform([limpar(texto)])
if pacote["nome"] == "GaussianNB":
    vetor = vetor.toarray()
classe = int(pacote["modelo"].predict(vetor)[0])
print("positivo" if classe == 1 else "negativo")

positivo
